# Mini-TP 3 — Sirve tu modelo por gRPC (starter)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual, esta semana**

Expón **tu propio modelo** (el mismo que serviste por REST en la Sesión 1) como un servicio **gRPC**. Este notebook es un *starter*: completa las celdas marcadas con `# TODO`.

**Qué debes entregar**
1. Un `scoring.proto` con un servicio para tu modelo (mensajes de entrada y salida **tipados**).
2. Los *stubs* generados y un **servidor** que cargue tu modelo **una sola vez**.
3. Un **cliente** que llame al servicio (**unary**).
4. Un método de **server-streaming** que puntúe un lote.
5. Una **comparación de latencia** contra tu endpoint REST de la Sesión 1, con una breve reflexión.

**Se evalúa:** que corra de punta a punta; que el `.proto` tipe entrada y salida; que funcionen unary y streaming; y la reflexión gRPC vs REST.

> Apóyate en `grpc_tutorial.ipynb`, que hace todo esto paso a paso. Entorno con uv:
> `uv add grpcio grpcio-tools scikit-learn joblib requests` (y lo que use tu modelo).

In [1]:
# Dependencias (equivalente uv: uv add grpcio grpcio-tools joblib ...)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "grpcio", "grpcio-tools", "scikit-learn", "joblib", "requests"])
print("Listo.")

Listo.


## 1. Tu modelo

Carga aquí **tu** modelo entrenado (por ejemplo `model.pkl`). Si todavía no lo tienes a mano, entrena uno rápido para probar la mecánica.

In [2]:
import sys
from pathlib import Path
import joblib

# El pipeline del TP1 usa transformadores propios (ImputadorBMIPorEdad, limpiar) que viven
# en mini-tps/tp1/preprocesamiento.py.
DIR_TP1 = (Path.cwd().parent / "tp1").resolve()
if not DIR_TP1.exists():
    raise FileNotFoundError(
        f"No encuentro {DIR_TP1}. Abri el notebook parado en mini-tps/tp3/."
    )
sys.path.insert(0, str(DIR_TP1))

CONTENIDO_JOBLIB = joblib.load(DIR_TP1 / "modelo" / "modelo.joblib")

pipeline = CONTENIDO_JOBLIB["pipeline"]
VERSION_MODELO = CONTENIDO_JOBLIB["version"]
UMBRAL = CONTENIDO_JOBLIB["umbral"]
FEATURES = CONTENIDO_JOBLIB["features"]

print("Modelo cargado |", "version:", VERSION_MODELO, "| umbral:", UMBRAL)
print("Features:", FEATURES)

Modelo cargado | version: 1.0.0 | umbral: 0.4
Features: ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status']


## 2. El contrato `.proto`

Define el servicio para tu modelo. Debe tener **al menos** un método unary y uno de streaming.

In [3]:
%%writefile scoring.proto
syntax = "proto3";
package scoring;

enum Genero {
  GENERO_NO_ESPECIFICADO = 0;
  GENERO_MALE = 1;
  GENERO_FEMALE = 2;
  GENERO_OTHER = 3;
}

enum EstadoCivil {
  ESTADO_CIVIL_NO_ESPECIFICADO = 0;
  ESTADO_CIVIL_YES = 1;
  ESTADO_CIVIL_NO = 2;
}

enum TipoTrabajo {
  TIPO_TRABAJO_NO_ESPECIFICADO = 0;
  TIPO_TRABAJO_PRIVATE = 1;
  TIPO_TRABAJO_SELF_EMPLOYED = 2;
  TIPO_TRABAJO_GOVT_JOB = 3;
  TIPO_TRABAJO_CHILDREN = 4;
  TIPO_TRABAJO_NEVER_WORKED = 5;
}

enum TipoResidencia {
  TIPO_RESIDENCIA_NO_ESPECIFICADO = 0;
  TIPO_RESIDENCIA_URBAN = 1;
  TIPO_RESIDENCIA_RURAL = 2;
}

enum EstadoTabaquismo {
  TABAQUISMO_NO_ESPECIFICADO = 0;
  TABAQUISMO_FORMERLY_SMOKED = 1;
  TABAQUISMO_NEVER_SMOKED = 2;
  TABAQUISMO_SMOKES = 3;
  TABAQUISMO_UNKNOWN = 4;
}

// --- Entrada: un paciente

message Caracteristicas {
  Genero gender = 1;
  double age = 2;
  int32 hypertension = 3;
  int32 heart_disease = 4;
  EstadoCivil ever_married = 5;
  TipoTrabajo work_type = 6;
  TipoResidencia Residence_type = 7;
  double avg_glucose_level = 8;
  optional double bmi = 9;
  EstadoTabaquismo smoking_status = 10;
}

// --- Entrada del server-streaming
// Lote
message LoteCaracteristicas {
  repeated Caracteristicas casos = 1;
}

// --- Salida

message Prediccion {
  int32 prediccion = 1;
  double probabilidad = 2;
  string version_modelo = 3;
  string descripcion = 4;
}

service Scoring {
  rpc Predecir(Caracteristicas) returns (Prediccion);              // unary: 1 -> 1
  rpc PredecirLote(LoteCaracteristicas) returns (stream Prediccion); // server streaming: 1 -> N
}

Overwriting scoring.proto


## 3. Genera los *stubs*

In [4]:
import sys, subprocess, os
subprocess.run([sys.executable, "-m", "grpc_tools.protoc",
                "-I.", "--python_out=.", "--grpc_python_out=.", "scoring.proto"], check=True)
print("Generados:", [f for f in os.listdir('.') if f.startswith('scoring_pb2')])

Generados: ['scoring_pb2.py', 'scoring_pb2_grpc.py']


## 4. El servidor

Implementa los métodos. Recuerda: el modelo se carga **una vez**.

In [ ]:
import grpc
import pandas as pd
from concurrent import futures
import scoring_pb2, scoring_pb2_grpc

# Se estableció que el .proto tipe los categoricos como enum, es por eso que se realiza
# la traducción a string.
MAPA_GENERO = {
    scoring_pb2.GENERO_MALE: "Male",
    scoring_pb2.GENERO_FEMALE: "Female",
    scoring_pb2.GENERO_OTHER: "Other",
}
MAPA_ESTADO_CIVIL = {
    scoring_pb2.ESTADO_CIVIL_YES: "Yes",
    scoring_pb2.ESTADO_CIVIL_NO: "No",
}
MAPA_TIPO_TRABAJO = {
    scoring_pb2.TIPO_TRABAJO_PRIVATE: "Private",
    scoring_pb2.TIPO_TRABAJO_SELF_EMPLOYED: "Self-employed",
    scoring_pb2.TIPO_TRABAJO_GOVT_JOB: "Govt_job",
    scoring_pb2.TIPO_TRABAJO_CHILDREN: "children",
    scoring_pb2.TIPO_TRABAJO_NEVER_WORKED: "Never_worked",
}
MAPA_TIPO_RESIDENCIA = {
    scoring_pb2.TIPO_RESIDENCIA_URBAN: "Urban",
    scoring_pb2.TIPO_RESIDENCIA_RURAL: "Rural",
}
MAPA_TABAQUISMO = {
    scoring_pb2.TABAQUISMO_FORMERLY_SMOKED: "formerly smoked",
    scoring_pb2.TABAQUISMO_NEVER_SMOKED: "never smoked",
    scoring_pb2.TABAQUISMO_SMOKES: "smokes",
    scoring_pb2.TABAQUISMO_UNKNOWN: "Unknown",
}
MAPAS = {
    "gender": MAPA_GENERO,
    "ever_married": MAPA_ESTADO_CIVIL,
    "work_type": MAPA_TIPO_TRABAJO,
    "Residence_type": MAPA_TIPO_RESIDENCIA,
    "smoking_status": MAPA_TABAQUISMO,
}

# Para validar los mensajes entrantes.
def validar(caso):
    problemas = []

    # Categoricos: el 0 del enum es el default, significa que el cliente no lo mando.
    for campo, mapa in MAPAS.items():
        valor = getattr(caso, campo)
        if valor not in mapa:
            problemas.append(f"{campo} sin especificar")

    # Chequeo numéricos
    if not 0 <= caso.age <= 100:
        problemas.append(f"age fuera de rango [0, 100]: {caso.age}")
    if caso.hypertension not in (0, 1):
        problemas.append(f"hypertension debe ser 0 o 1: {caso.hypertension}")
    if caso.heart_disease not in (0, 1):
        problemas.append(f"heart_disease debe ser 0 o 1: {caso.heart_disease}")
    if not 0 < caso.avg_glucose_level <= 400:
        problemas.append(f"avg_glucose_level fuera de rango (0, 400]: {caso.avg_glucose_level}")
    if caso.HasField("bmi") and not 0 < caso.bmi <= 100:
        problemas.append(f"bmi fuera de rango (0, 100]: {caso.bmi}")

    return problemas

# Mensaje caracteristicas -> dataFrame de una fila, con las columnas que espera el pipeline.
def a_dataframe(caso):
    fila = {
        "gender": MAPA_GENERO[caso.gender],
        "age": caso.age,
        "hypertension": caso.hypertension,
        "heart_disease": caso.heart_disease,
        "ever_married": MAPA_ESTADO_CIVIL[caso.ever_married],
        "work_type": MAPA_TIPO_TRABAJO[caso.work_type],
        "Residence_type": MAPA_TIPO_RESIDENCIA[caso.Residence_type],
        "avg_glucose_level": caso.avg_glucose_level,
        "bmi": caso.bmi if caso.HasField("bmi") else None,
        "smoking_status": MAPA_TABAQUISMO[caso.smoking_status],
    }
    return pd.DataFrame([fila], columns=FEATURES)


def puntuar(caso):
    probabilidad = float(pipeline.predict_proba(a_dataframe(caso))[0, 1])
    clase = int(probabilidad >= UMBRAL)
    return scoring_pb2.Prediccion(
        prediccion=clase,
        probabilidad=probabilidad,
        version_modelo=VERSION_MODELO,
        descripcion="Tiene riesgo de ACV" if clase == 1 else "No tiene riesgo de ACV",
    )


class ScoringServicer(scoring_pb2_grpc.ScoringServicer):
    # Unary: 1 caso -> 1 prediccion.
    def Predecir(self, request, context):
        problemas = validar(request)
        if problemas:
            context.abort(grpc.StatusCode.INVALID_ARGUMENT, "; ".join(problemas))
        return puntuar(request)
    #Server streaming: 1 lote -> N predicciones
    def PredecirLote(self, request, context):
        for i, caso in enumerate(request.casos):
            problemas = validar(caso)
            if problemas:
                context.abort(grpc.StatusCode.INVALID_ARGUMENT, f"caso {i}: " + "; ".join(problemas))
            yield puntuar(caso)


servidor = grpc.server(futures.ThreadPoolExecutor(max_workers=4))
scoring_pb2_grpc.add_ScoringServicer_to_server(ScoringServicer(), servidor)
servidor.add_insecure_port("[::]:50051")
servidor.start()
print("Servidor gRPC en localhost:50051")

Servidor gRPC en localhost:50051


## 5. El cliente (unary)

In [6]:
canal = grpc.insecure_channel("localhost:50051")
stub = scoring_pb2_grpc.ScoringStub(canal)

# CASO_VALIDO de cliente_prueba.py
paciente = scoring_pb2.Caracteristicas(
    gender=scoring_pb2.GENERO_MALE,
    age=67.0,
    hypertension=0,
    heart_disease=1,
    ever_married=scoring_pb2.ESTADO_CIVIL_YES,
    work_type=scoring_pb2.TIPO_TRABAJO_PRIVATE,
    Residence_type=scoring_pb2.TIPO_RESIDENCIA_URBAN,
    avg_glucose_level=228.69,
    bmi=36.6,
    smoking_status=scoring_pb2.TABAQUISMO_FORMERLY_SMOKED,
)

# Prueba caso válido
respuesta = stub.Predecir(paciente)
print("prediccion :", respuesta.prediccion)
print("probabilidad:", round(respuesta.probabilidad, 4))
print("version    :", respuesta.version_modelo)
print("->", respuesta.descripcion)

# Prueba caso incompleto (categorico sin especificar)
incompleto = scoring_pb2.Caracteristicas()
incompleto.CopyFrom(paciente)
incompleto.ClearField("gender")
try:
    stub.Predecir(incompleto)
except grpc.RpcError as error:
    print("\nsin gender -> rechazado |", error.code().name, "|", error.details())

# Prueba caso inválido: CASO_INVALIDO de cliente_prueba.py.
invalido = scoring_pb2.Caracteristicas()
invalido.CopyFrom(paciente)
invalido.age = 200
invalido.avg_glucose_level = -5.0
try:
    stub.Predecir(invalido)
except grpc.RpcError as error:
    print("caso invalido -> rechazado |", error.code().name, "|", error.details())

prediccion : 1
probabilidad: 0.4752
version    : 1.0.0
-> Tiene riesgo de ACV

sin gender -> rechazado | INVALID_ARGUMENT | gender sin especificar
caso invalido -> rechazado | INVALID_ARGUMENT | age fuera de rango [0, 100]: 200.0; avg_glucose_level fuera de rango (0, 400]: -5.0


## 6. Streaming

Una vez que agregaste `PredictStream` al `.proto`, lo regeneraste y lo implementaste en el servidor, pruébalo aquí.

In [7]:
import time

# Lote de 3 pacientes: el de arriba, el mismo sin bmi y uno joven de bajo riesgo.
sin_bmi = scoring_pb2.Caracteristicas()
sin_bmi.CopyFrom(paciente)
sin_bmi.ClearField("bmi")

joven = scoring_pb2.Caracteristicas()
joven.CopyFrom(paciente)
joven.age = 22.0
joven.avg_glucose_level = 85.0
joven.heart_disease = 0
joven.smoking_status = scoring_pb2.TABAQUISMO_NEVER_SMOKED

lote = scoring_pb2.LoteCaracteristicas(casos=[paciente, sin_bmi, joven])

inicio = time.perf_counter()
for i, p in enumerate(stub.PredecirLote(lote)):
    t = (time.perf_counter() - inicio) * 1000
    print(f"caso {i} (+{t:6.2f} ms) | prediccion={p.prediccion} "
          f"probabilidad={p.probabilidad:.4f} -> {p.descripcion}")

caso 0 (+ 10.83 ms) | prediccion=1 probabilidad=0.4752 -> Tiene riesgo de ACV
caso 1 (+ 19.47 ms) | prediccion=1 probabilidad=0.5587 -> Tiene riesgo de ACV
caso 2 (+ 25.73 ms) | prediccion=0 probabilidad=0.0234 -> No tiene riesgo de ACV


## 7. Comparación de latencia gRPC vs REST

Levanta (o apunta a) tu REST de la Sesión 1 y mide el tiempo medio por llamada. Anota abajo tu reflexión.

In [8]:
import threading

import requests
import uvicorn

# api.py de TP1
import api

PUERTO_REST = 8001
URL_REST = f"http://127.0.0.1:{PUERTO_REST}/v1/predict"

def _run():
    uvicorn.run(api.app, host="127.0.0.1", port=PUERTO_REST, log_level="error")

threading.Thread(target=_run, daemon=True).start()
time.sleep(1.5) 
print("API REST del TP1 en", URL_REST)

payload = {
    "gender": "Male",
    "age": 67.0,
    "hypertension": 0,
    "heart_disease": 1,
    "ever_married": "Yes",
    "work_type": "Private",
    "Residence_type": "Urban",
    "avg_glucose_level": 228.69,
    "bmi": 36.6,
    "smoking_status": "formerly smoked",
}

N = 200

# REST
t = time.perf_counter()
for _ in range(N):
    requests.post(URL_REST, json=payload)
rest_ms = (time.perf_counter() - t) / N * 1000


t = time.perf_counter()
for _ in range(N):
    stub.Predecir(paciente)
grpc_ms = (time.perf_counter() - t) / N * 1000

print(f"REST : {rest_ms:.3f} ms/llamada")
print(f"gRPC : {grpc_ms:.3f} ms/llamada")
print(f"gRPC es ~{rest_ms/grpc_ms:.1f}x más rápido en esta prueba local")


API REST del TP1 en http://127.0.0.1:8001/v1/predict
REST : 8.190 ms/llamada
gRPC : 6.019 ms/llamada
gRPC es ~1.4x más rápido en esta prueba local


### Reflexión

En la prueba local, gRPC fue aproximadamente 1.4 veces más rápido que REST. También se comprobó que el contrato definido permite validar y estructurar los datos recibidos.

Actualmente, en una arquitectura como la de MLOps I, el frontend se comunica por REST con FastAPI, y FastAPI carga el modelo champion desde MLflow y la metadata desde MinIO para realizar la predicción. Se podría mantener REST para la comunicación externa con el frontend y separar la inferencia en un nuevo servicio interno de predicción, al que FastAPI llamaría mediante gRPC. Ese servicio usaría el modelo champion obtenido desde MLflow para realizar la inferencia interna.

Como costo, gRPC requiere definir y mantener el contrato .proto.

In [9]:
# limpieza
canal.close(); servidor.stop(0)
print("Servidor detenido.")

Servidor detenido.
